# Core-satellite trend portfolio for a family office
**2270 Financial Modeling, Group Assignment**

Moodle group: **TO BE COMPLETED** · Members: **TO BE COMPLETED**  
GitHub: https://github.com/nilscuypers18-ops/FM-Group-Project · Streamlit app: **TO BE COMPLETED**

**How to run:** open this notebook in a fresh Google Colab session and choose *Runtime → Run all*.
All data come from the Yahoo Finance API; no other files are needed.

Cells that start with `# STRATEGY FUNCTIONS` hold the model. The Streamlit app (`app.py`)
reads exactly these cells from this notebook, so the app always uses the same calculations.

**What this cell does:** installs `yfinance`, the package that downloads prices from Yahoo Finance. Colab has the other packages already.

In [ ]:
!pip install -q yfinance

**What this cell does:** loads the packages we use (NumPy for maths, pandas for tables, Matplotlib for charts) and sets a common chart style.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3})

## 0. Client, investment rule and assumptions
**Client:** a conservative USD family office with a long horizon that wants to preserve and grow wealth.

* **Core:** 60% AGG (US aggregate bonds), held at a fixed weight. AGG carries interest-rate and credit risk; it is not cash.
* **Satellite:** 40% split across SPY (US equity), GLD (gold) and DBC (commodities) in proportion to 1 / (12-month volatility).
* **Trend filter:** at each month-end, a satellite asset is *on* if its price is above its 10-month simple moving average (SMA). If it is *off*, its share goes to 3-month T-bills.
* **Timing:** the signal uses the month-end close of month t; the new weights earn the return of month t+1 (no look-ahead, Lecture 4).
* **Rebalancing and costs:** monthly; 10 bps × traded weight (buys plus sells).
* **Benchmark:** 40/60 SPY/AGG (40% US equity, 60% US bonds), rebalanced monthly. It has the same 60% bond share as our core, so the comparison shows what our diversified, trend-filtered 40% satellite adds compared with simply holding 40% in SPY.
* **Sample:** prices from February 2006 (DBC starts trading then). The first full 12-month volatility is available in February 2007, so **returns run from March 2007 to October 2026**.

**What this cell does:** stores every assumption in one place: the tickers, the sample dates, the 60% core weight, the 10-month moving average, the 12-month volatility window and the 10 bps cost. Change a number here and the whole notebook (and the app) uses the new value.

In [ ]:
# STRATEGY FUNCTIONS: app.py loads every cell that starts with this line
import numpy as np
import pandas as pd

CORE = "AGG"
SATELLITE = ["SPY", "GLD", "DBC"]
ASSETS = [CORE] + SATELLITE
TBILL = "TBILL"
RF_TICKER = "^IRX"                 # 13-week T-bill yield, % per year

DOWNLOAD_START = "2006-02-01"
BACKTEST_START = "2007-03-31"      # first month with a portfolio return
BACKTEST_END = "2026-10-31"        # last month with a portfolio return

CORE_WEIGHT = 0.60
MA_MONTHS = 10
VOL_MONTHS = 12
COST_BPS = 10
BENCHMARK = "40/60 SPY/AGG"
BENCHMARK_WEIGHTS = {"SPY": 0.40, "AGG": 0.60}   # 40% equity, 60% bonds, rebalanced monthly

## 1. Data
Source: Yahoo Finance via `yfinance` (Lecture 3). Daily adjusted closes (dividends and splits included), USD,
converted to the last trading day of each month. `^IRX` is a quoted annual yield in percent; the cash return
in month t is the yield at the end of month t−1 divided by 100 and 12 (an approximation of a T-bill return).

**What this cell does:** defines `download_data()`. It downloads daily prices for AGG, SPY, GLD, DBC and the T-bill yield, keeps the last price of each month, and turns the T-bill yield into a monthly cash return. If any month is missing it stops with an error instead of guessing.

In [ ]:
# STRATEGY FUNCTIONS: app.py loads every cell that starts with this line
def download_data(start=DOWNLOAD_START, end=BACKTEST_END):
    """Month-end prices and monthly T-bill returns from Yahoo Finance."""
    import yfinance as yf
    end_exclusive = pd.Timestamp(end) + pd.Timedelta(days=1)   # yfinance's end date is exclusive
    daily = yf.download(ASSETS + [RF_TICKER], start=start, end=end_exclusive,
                        auto_adjust=True, progress=False)["Close"]
    monthly = daily.resample("ME").last()                      # last trading day of each month
    if monthly.isna().any().any():
        raise ValueError("Missing month-end data. Inspect the download before continuing.")
    prices = monthly[ASSETS]
    rf = (monthly[RF_TICKER].shift(1) / 100 / 12).rename(TBILL)
    return prices, rf

**What this cell does:** runs the download and performs the four data checks from Lecture 3: how many rows and columns, which dates, whether values are missing, and a plot. The plot divides every price by its first value so all assets start at 1 and can be compared.

In [ ]:
prices, rf = download_data()

# The four checks from Lecture 3: shape, dates, missing values, plot
print("Shape:", prices.shape)
print("Dates:", prices.index[0].date(), "to", prices.index[-1].date())
print("Missing values:", int(prices.isna().sum().sum()))
(prices / prices.iloc[0]).plot(title="Month-end prices (first month = 1)")
plt.show()

**What this cell does:** calculates monthly returns (price this month / price last month − 1) and adds the T-bill return as a fifth asset. It then shows each asset's average return and volatility per year, and how strongly the assets move together (correlation). Low correlations are the reason for combining bonds, equities, gold and commodities.

In [ ]:
returns = prices.pct_change()        # simple monthly returns
returns[TBILL] = rf                   # add the T-bill as the cash asset
print("Annualized mean and volatility of monthly returns:")
# x12 for the mean and x sqrt(12) for volatility (Lecture 3)
display(pd.DataFrame({"Mean": returns.mean() * 12, "Volatility": returns.std() * np.sqrt(12)}).round(3))
print("Correlation of monthly returns:")
display(returns[ASSETS].corr().round(2))

We use **simple returns**, not log returns as in the lecture examples: a portfolio's return is the weighted
sum of its assets' *simple* returns, which does not hold for log returns.

## 2. Signals and weights
The trend signal is the Lecture 4 SMA rule on monthly prices. Satellite shares are inverse 12-month volatilities,
normalized to sum to one. If every satellite asset is off, the portfolio holds 60% AGG and 40% T-bills.

**What this cell does:** defines the three building blocks of the allocation rule:
* `trend_signal()`: 1 (on) if the price is above its 10-month average, else 0 (off).
* `rolling_vol()`: each asset's volatility over the last 12 months.
* `strategy_weights()`: gives AGG 60%, splits the 40% satellite by 1/volatility, sets off assets to 0 and puts their share into T-bills.

In [ ]:
# STRATEGY FUNCTIONS: app.py loads every cell that starts with this line
def trend_signal(prices, ma_months=MA_MONTHS):
    """1 if the month-end price is above its moving average, else 0."""
    sma = prices.rolling(ma_months).mean()
    return (prices > sma).astype(float).where(sma.notna())


def rolling_vol(prices, vol_months=VOL_MONTHS):
    """Annualized volatility of the last `vol_months` monthly returns."""
    return prices.pct_change().rolling(vol_months).std() * np.sqrt(12)


def strategy_weights(prices, core_weight=CORE_WEIGHT, ma_months=MA_MONTHS, vol_months=VOL_MONTHS):
    """Target weights decided at each month-end."""
    signal = trend_signal(prices[SATELLITE], ma_months)
    inv_vol = 1 / rolling_vol(prices[SATELLITE], vol_months)
    share = inv_vol.div(inv_vol.sum(axis=1), axis=0)          # satellite shares sum to 1
    weights = pd.DataFrame(index=prices.index)
    weights[CORE] = core_weight
    weights[SATELLITE] = (1 - core_weight) * share * signal   # 0 if the asset is off
    weights[TBILL] = (1 - core_weight) - weights[SATELLITE].sum(axis=1)
    ready = signal.notna().all(axis=1) & share.notna().all(axis=1)
    return weights[ready].clip(lower=0)

**What this cell does:** calculates the target weights for every month and shows an example for SPY: the price with its moving average (top) and the resulting on/off signal (bottom). The table shows the weights of the last five months; each row sums to 100%.

In [ ]:
weights = strategy_weights(prices)               # target weights for every month
sma = prices["SPY"].rolling(MA_MONTHS).mean()
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
pd.DataFrame({"SPY": prices["SPY"], f"{MA_MONTHS}-month SMA": sma}).loc["2018":"2023"].plot(ax=ax[0], title="Example: SPY and its moving average")
trend_signal(prices[["SPY"]]).loc["2018":"2023"].plot(ax=ax[1], drawstyle="steps-post", title="SPY signal (1 = on)", legend=False)
plt.tight_layout()
plt.show()
print("First decision date:", weights.index[0].date())
weights.tail()

## 3. Backtest
The key line is `shift(1)`: weights decided at the end of month t earn the returns of month t+1 (Lecture 4).
Between rebalancing dates the weights drift with returns, so turnover is measured against the drifted weights.
Cost in a month = 10 bps × turnover, subtracted from that month's return (net = gross − cost).
The first month counts as buying the whole portfolio (turnover = 1). The benchmark follows the same conventions.

**What this cell does:** defines the backtest.
* `backtest()` goes month by month: it takes the weights decided last month (`shift(1)`), earns this month's returns, counts how much was traded (turnover) and subtracts the cost. Between trades the weights drift with the returns.
* `run_all()` runs two portfolios on the same dates: our strategy and the 40/60 SPY/AGG benchmark (rebalanced monthly).

In [ ]:
# STRATEGY FUNCTIONS: app.py loads every cell that starts with this line
def backtest(target_weights, returns, start=BACKTEST_START, end=BACKTEST_END,
             cost_bps=COST_BPS, rebalance=True):
    """Monthly backtest with drifting weights and transaction costs."""
    positions = target_weights.reindex(returns.index).shift(1)   # decided at t, held in t+1
    rets = returns.loc[start:end, target_weights.columns]
    if positions.loc[rets.index].isna().any().any() or rets.isna().any().any():
        raise ValueError("Missing weights or returns in the backtest period.")
    held = pd.Series(0.0, index=target_weights.columns)          # start from cash
    rows = []
    for month, r in rets.iterrows():
        new = positions.loc[month] if (rebalance or not rows) else held
        turnover = (new - held).abs().sum()
        gross = (new * r).sum()
        cost = turnover * cost_bps / 10_000
        rows.append({"date": month, "gross": gross, "cost": cost, "net": gross - cost,
                     "turnover": turnover, **{f"weight_{a}": new[a] for a in new.index},
                     **{f"pretrade_{a}": held[a] for a in new.index},      # drifted weights before trading
                     **{f"trade_{a}": new[a] - held[a] for a in new.index}})  # used by the trade history page
        held = new * (1 + r) / (1 + gross)                       # weights drift until next trade
    return pd.DataFrame(rows).set_index("date")


def run_all(prices, rf, core_weight=CORE_WEIGHT, ma_months=MA_MONTHS, cost_bps=COST_BPS,
            start=BACKTEST_START, end=BACKTEST_END):
    """Strategy and the 40/60 SPY/AGG benchmark. Returns (net returns, rf, weights, strategy detail)."""
    returns = prices.pct_change()
    returns[TBILL] = rf
    weights = strategy_weights(prices, core_weight, ma_months)
    fixed = lambda w: pd.DataFrame(w, index=returns.index)
    detail = backtest(weights, returns, start, end, cost_bps)
    results = pd.DataFrame({
        "Strategy": detail["net"],
        BENCHMARK: backtest(fixed(BENCHMARK_WEIGHTS), returns, start, end, cost_bps)["net"],
    })
    return results, rf.loc[results.index], weights, detail

## 4. Performance measures

**What this cell does:** defines the performance measures.
* `cumulative()`: how USD 1 grows over time, starting at 1.
* `drawdowns()`: how far the portfolio is below its previous peak, in %.
* `performance_table()`: annualized return, CAGR, volatility, Sharpe ratio, alpha, beta, tracking error, information ratio and maximum drawdown, using the formulas in the table below.

In [ ]:
# STRATEGY FUNCTIONS: app.py loads every cell that starts with this line
def cumulative(returns):
    """Growth of 1, starting at 1 the month before the first return."""
    wealth = (1 + returns).cumprod()
    start = pd.DataFrame(1.0, index=[wealth.index[0] - pd.offsets.MonthEnd(1)],
                         columns=wealth.columns) if isinstance(wealth, pd.DataFrame) \
        else pd.Series(1.0, index=[wealth.index[0] - pd.offsets.MonthEnd(1)])
    return pd.concat([start, wealth])


def drawdowns(returns):
    """Loss from the high water mark (positive number), including initial wealth 1."""
    wealth = cumulative(returns)
    return (1 - wealth / wealth.cummax()).iloc[1:]


def performance_table(results, rf, benchmark=BENCHMARK):
    """Lecture 3 formulas on monthly returns. Beta is not annualized."""
    bench_excess = results[benchmark] - rf
    rows = {}
    for name, r in results.items():
        excess = r - rf
        beta = excess.cov(bench_excess) / bench_excess.var()
        active = r - results[benchmark]
        tracking_error = active.std() * np.sqrt(12)
        rows[name] = {
            "Ann. return": r.mean() * 12,
            "CAGR": (1 + r).prod() ** (12 / len(r)) - 1,
            "Volatility": r.std() * np.sqrt(12),
            "Sharpe": excess.mean() * 12 / (excess.std() * np.sqrt(12)),
            "Alpha (ann.)": (excess.mean() - beta * bench_excess.mean()) * 12,
            "Beta": beta,
            "Tracking error": tracking_error,
            "Information ratio": active.mean() * 12 / tracking_error if tracking_error > 0 else np.nan,
            "Max drawdown": drawdowns(r).max(),
        }
    return pd.DataFrame(rows).T

**What this cell does:** runs the backtest for the strategy and the benchmark and draws the two main charts: growth of USD 1 after costs (both start at 1) and the weights the strategy actually held each month.

In [ ]:
results, rf_bt, weights, detail = run_all(prices, rf)
# weights actually held in each month (after drift)
held = detail.filter(like="weight_").rename(columns=lambda c: c.replace("weight_", ""))

cumulative(results).plot(title="Cumulative performance, net of costs (start = 1)")
plt.ylabel("Value of USD 1")
plt.show()
held.plot.area(title="Weights held during each month", ylim=(0, 1))
plt.legend(loc="upper left", ncol=5)
plt.show()

| Measure | Formula (monthly returns r, benchmark b, T-bill rf) |
|---|---|
| Annualized return | 12 × mean(r) |
| Volatility | √12 × std(r) |
| Sharpe ratio | 12 × mean(r − rf) / (√12 × std(r − rf)) |
| Beta (not annualized) | cov(r − rf, b − rf) / var(b − rf) |
| Alpha (annualized) | 12 × [mean(r − rf) − beta × mean(b − rf)] |
| Information ratio | 12 × mean(r − b) / (√12 × std(r − b)) |
| Maximum drawdown | max(1 − wealth / high water mark) |

The benchmark is the 40/60 SPY/AGG portfolio; its own information ratio is undefined.

**What this cell does:** prints the performance table for the prospectus, plus two implementation numbers: how much of the portfolio is traded per month on average, and how much USD 1 grows before vs. after transaction costs.

In [ ]:
table = performance_table(results, rf_bt)
pct = ["Ann. return", "CAGR", "Volatility", "Alpha (ann.)", "Tracking error", "Max drawdown"]
display(table.style.format({c: "{:.2%}" for c in pct} | {c: "{:.2f}" for c in ["Sharpe", "Beta", "Information ratio"]},
                           na_rep="–"))
print(f"Average monthly turnover (after the first month): {detail['turnover'].iloc[1:].mean():.1%}")
print(f"Growth of USD 1, gross vs net of costs: {(1 + detail['gross']).prod():.2f} vs {(1 + detail['net']).prod():.2f}")

**What this cell does:** plots the drawdowns (loss from the previous peak) and shows the total return of each portfolio in three stress periods: the financial crisis, the Covid crash and the 2022 stock and bond selloff.

In [ ]:
drawdowns(results).plot(title="Drawdown: loss from the high water mark")
plt.ylabel("Loss from peak")
plt.show()

# total return in each stress period = product of (1 + monthly return) - 1
crises = {"Financial crisis (Mar 2007 – Feb 2009)": ("2007-03", "2009-02"),
          "Covid crash (Feb – Mar 2020)": ("2020-02", "2020-03"),
          "Stock and bond selloff (2022)": ("2022-01", "2022-12")}
display(pd.DataFrame({name: (1 + results.loc[a:b]).prod() - 1 for name, (a, b) in crises.items()}).T
        .style.format("{:.1%}"))

## 5. Transaction costs and the moving-average length
We keep the base rule fixed and only check how sensitive the result is (Lecture 4, transaction costs).
This is not a search for the best parameter.

**What this cell does:** reruns the strategy with different transaction costs (0, 10, 25, 50 bps) and moving-average lengths (8 and 12 months), to show how much the result depends on these assumptions.

In [ ]:
# same rule, only one assumption changed at a time
rows = {}
for cost in [0, 10, 25, 50]:
    rows[f"MA 10, {cost} bps"] = performance_table(*run_all(prices, rf, cost_bps=cost)[:2]).loc["Strategy"]
for ma in [8, 12]:
    rows[f"MA {ma}, 10 bps"] = performance_table(*run_all(prices, rf, ma_months=ma)[:2]).loc["Strategy"]
display(pd.DataFrame(rows).T[["Ann. return", "Volatility", "Sharpe", "Max drawdown"]]
        .style.format({"Ann. return": "{:.2%}", "Volatility": "{:.2%}", "Sharpe": "{:.2f}", "Max drawdown": "{:.2%}"}))

## 6. Limitations
* **Execution:** trades are assumed at the month-end close that produced the signal. This is optimistic; spreads and market impact are not modelled beyond the 10 bps cost.
* **Hindsight:** the ETFs were chosen today. The backtest is one historical path, not an out-of-sample test.
* **Cash proxy:** `^IRX / 100 / 12` approximates the T-bill return; it is not an investable index.
* **Risks for the client:** the 60% bond core has interest-rate and credit risk (see 2022); trend filters react with a delay and can whipsaw.
* **Not modelled:** taxes, inflation, management fees beyond ETF expense ratios.

## References and AI disclosure
Course notebooks: Lecture 3 (APIs and pandas; portfolio strategies), Lecture 4 (backtesting), 2270 formula sheet.
Data: Yahoo Finance via `yfinance`.

**AI disclosure (group to complete):** list every AI tool used (e.g. Claude, OpenAI Codex), what it was used for,
and how the group checked the output.